# Tutorial: Queueing Work Behind an HTTP API

A user requests a report. Creating it takes a few seconds. Instead of making the user wait
for the original HTTP request, our API accepts the job and returns an ID. The user can
then check whether the report is ready.

In this tutorial you will run Redis, a worker, and a FastAPI application in your **GitHub
Codespace**. The scripts are already in this folder. You will start them in terminals and
use the notebook to send HTTP requests. No external API keys are needed.

At the end, you will apply the same approach to the existing Docker Compose image lab.

## 1. What are the queue and worker for?

- The **API** accepts an HTTP request and creates a job.
- **Redis** is a data store that keeps data in memory for fast access. Here, it holds
  the queue, job states, and results. As a separate server, it lets the API and worker
  share this data even though they run in different processes.
- The **worker** takes a job from the queue and runs the Python function.
- The **client** uses the job ID to ask the API for its status and result.

We use **RQ**, a Python library, to connect our application to the queue in Redis.
Redis stores the work; the worker executes it.

![An HTTP request becomes a queued job, which a worker processes](figures/queue-flow.svg)

This separation lets the API respond quickly while work continues independently. Jobs can
wait while workers are busy. More workers can process more jobs at once, but a queue does
not make an individual job faster.

## 2. Install the dependencies in Codespaces

Open this repository in its GitHub Codespace. Python and Docker are provided by the
repository's development container.

In a terminal, from the repository root:

```bash
cd optional-lab-redis
python -m pip install -r requirements.txt
```

Keep this notebook open. For each new terminal below, first enter `optional-lab-redis`
from the repository root.

## 3. Start Redis — terminal 1

Run:

```bash
docker run --rm --name redis-tutorial -p 6379:6379 redis:7.4-alpine
```

Leave this terminal running. The Python scripts in the Codespace can now reach Redis at
`localhost:6379`. If a Redis server is already using that port, stop the previous demo
before starting this one.

## 4. Submit a job without an HTTP API — terminal 2

Open [`tasks.py`](tasks.py). Its `build_report` function waits briefly and returns a report.
The wait stands in for something slow, such as generating an image or exporting data.

Open a second terminal, enter `optional-lab-redis`, and run:

```bash
python enqueue.py "First report"
```

This runs [`enqueue.py`](enqueue.py), which submits the function call to the queue:

```python
# Submit work now; a worker will run the function later.
job = queue.enqueue(build_report, "First report")
print(job.id)
```

The script prints a job ID and exits. **The report is not ready yet:** no worker is running.
Redis is holding the job until a worker takes it.

## 5. Start the worker — terminal 2

In the same terminal, run:

```bash
python worker.py
```

Leave it running. Watch the log: the worker takes the waiting job and completes it.
[`worker.py`](worker.py) listens to the same `reports` queue used by `enqueue.py`.

You have now submitted and processed a job without FastAPI. Next, the API will submit
jobs in exactly the same way when it receives an HTTP request.

## 6. Start FastAPI — terminal 3

Open a third terminal, enter `optional-lab-redis`, and run:

```bash
python -m uvicorn app:app --host 0.0.0.0 --port 8000
```

Leave this terminal running too. The application is in [`app.py`](app.py).
In the Codespaces **Ports** tab, open the forwarded port **8000** and add `/docs` to the
browser URL to try the endpoints interactively.

The notebook runs inside the Codespace, so its requests can use `http://localhost:8000`.
At this point you have three running processes: Redis, the worker, and the API.

## 7. Compare a direct request with a queued request

The API offers two ways to create the same report:

| Request | What the response means |
|---|---|
| `POST /reports/sync` | The report has been created; here is the result |
| `POST /reports` | The job has been accepted; here is its ID and status URL |

Run the next two cells and compare their response times and bodies.

In [ ]:
import time
import httpx

API_URL = "http://localhost:8000"
report_request = {"name": "Monthly report", "delay": 2}

# This request waits until the report has been created.
started = time.perf_counter()
response = httpx.post(f"{API_URL}/reports/sync", json=report_request, timeout=10)
response.raise_for_status()
print("HTTP", response.status_code, "after", round(time.perf_counter() - started, 2), "seconds")
print(response.json())

In [ ]:
# This request only submits the job. The worker creates the report separately.
started = time.perf_counter()
response = httpx.post(f"{API_URL}/reports", json=report_request, timeout=10)
response.raise_for_status()
job = response.json()
print("HTTP", response.status_code, "after", round(time.perf_counter() - started, 2), "seconds")
print(job)
print("Location:", response.headers["Location"])

The direct request waits about two seconds and returns the report. The queued request
returns quickly with **202 Accepted**, a `job_id`, and a `status_url`.

**202 does not mean the report is ready.** It means the API has accepted the work.
The `Location` header points to the status endpoint too.

![POST a job, receive 202 and a job ID, then GET its status and result](figures/http-job-lifecycle.svg)

## 8. Ask for the job status

Use the returned URL to check the job. The API returns one of these states:

| Status | Meaning |
|---|---|
| `queued` | Waiting for a worker |
| `running` | A worker is processing the job |
| `succeeded` | The result is ready |
| `failed` | Processing failed |

A status lookup returns **HTTP 200** with that state in its JSON body. The HTTP request
succeeded even if the job itself failed.

In [ ]:
response = httpx.get(f"{API_URL}{job['status_url']}", timeout=10)
response.raise_for_status()
print(response.json())

The job may already be finished by the time you run the cell. That is fine: the client
does not have to see every intermediate state.

To keep checking automatically, the client can **poll**: send a status request, wait a
little, then try again. Stop when the job succeeds or fails. Always limit how long the
client waits.

In [ ]:
# Check at most 30 times, with a short pause between requests.
# Polling checks the job; it does not submit it again.
for _ in range(30):
    response = httpx.get(f"{API_URL}{job['status_url']}", timeout=10)
    response.raise_for_status()
    status = response.json()
    print(status)
    if status["status"] in {"succeeded", "failed"}:
        break
    time.sleep(0.5)
else:
    print("Still waiting. Check whether the worker is running.")

## 9. Try a job that fails

The example task deliberately fails when its name is `FAIL`. Submit it below while the
worker is running, then run the polling cell from section 8 again.

In [ ]:
response = httpx.post(
    f"{API_URL}/reports", json={"name": "FAIL", "delay": 1}, timeout=10
)
response.raise_for_status()
job = response.json()
print("Accepted:", response.status_code, job)

The POST returns **202**, but the eventual job state is **failed**. The client should
show the error and stop polling rather than display a loading indicator forever.

Other HTTP responses to consider:
- **404:** The requested job ID is unknown or no longer available.
- **422:** The submitted data is invalid, such as an empty report name.
- **503:** Redis is unavailable, so the API could not accept the job.

A queue does not guarantee that every job succeeds. Repeating a POST also creates a new
job, so retries must be deliberate. Closing the browser or stopping polling does not
cancel work that was already accepted.

When you are done, press **Ctrl+C** in the API and worker terminals and run:

```bash
docker stop redis-tutorial
```

## 10. Exercise: Add Redis to Your Docker Compose Lab

Apply this flow to your existing [image-generation lab](../lab-assignments.ipynb).
Extend [`docker-lab/docker-compose.yml`](../docker-lab/docker-compose.yml) so that the
frontend, backend, Redis, and worker run together.

![Frontend, backend, Redis, worker, and shared image storage](figures/compose-architecture.svg)

The code already contains the queue-based image endpoint:
- `POST /v3/images` accepts work and returns an `img_id`.
- `GET /images/{img_id}` returns **202** while the image is missing, then **200** with the JPEG.
- The existing `backend/worker.py` processes jobs from the `default` queue.
- Select **v3** in Streamlit to use this path.

The image API returns **200 with an ID on POST**, unlike the **202** used by our report
example. Both separate acceptance from completion; the client must follow the actual
API's contract. Keep the existing image endpoints for this exercise.

### Connect the services

Add a `redis` service and a `worker` service alongside your frontend and backend. The
worker can use the backend image with the command `python worker.py`.

| Service | Configuration |
|---|---|
| Frontend | `API_URL=http://backend:8000` |
| Backend and worker | `REDIS_URL=redis://redis:6379/0` |
| Backend and worker | The same `UPLOAD_FOLDER` and shared image volume |
| Redis | A healthcheck using `redis-cli ping` |

Inside Compose, use service names to reach other containers. `localhost` inside one
container does not refer to the Redis container next to it.

The worker writes the image and the backend reads it, so both need the same writable
image directory. Redis holds the job, not the JPEG. Ensure the directory exists.
The backend and worker should wait until Redis is healthy before starting.

Unlike the report tutorial, the image lab requires valid provider credentials and credits.
Provide these through runtime configuration and keep them out of submission screenshots.

### Test and submit your solution

From `docker-lab/`:

1. Start the stack with `docker compose up --build -d`.
2. Submit an image request through Streamlit v3 or `POST /v3/images`. Use the returned ID
   to retrieve the finished image.
3. Follow the worker logs with `docker compose logs -f worker` to see the job being processed.
4. Submit several requests. Explain why the API can return IDs while the worker is still
   processing images. Retrieve each image using its ID, without sending another POST.

Submit your Compose file, any required Dockerfile changes, and evidence of those requests
and responses. Briefly explain what the API, Redis, and worker each do.

**Discuss:** The existing image endpoint keeps returning 202 when the file is missing,
including when a job has failed. How would you improve the status response so the client
can stop waiting and show an error?

**Optional:** Try `docker compose up -d --scale worker=2`. Does finishing order always
match submission order? More workers increase processing capacity, not the speed of each
individual job.

## References

- [RQ: submitting jobs and running workers](https://python-rq.org/docs/)
- [FastAPI: background tasks and separate workers](https://fastapi.tiangolo.com/tutorial/background-tasks/)
- [Docker Compose networking](https://docs.docker.com/compose/how-tos/networking/)

The scripts use the dependencies in `requirements.txt`. The diagrams are stored in `figures/`.